# 03 - Facebook Prophet Model

> Prophet is an additive time series model developed by Meta that handles yearly/weekly seasonality, holidays, and trend changepoints automatically.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
COLORS = {'primary':'#003B73','alert':'#BF212F','success':'#27AE60'}
print("Libraries loaded")

## 1. Load Prepared Data

In [ ]:
train = pd.read_csv('../data/train_ts.csv', parse_dates=['ds'])
test = pd.read_csv('../data/test_ts.csv', parse_dates=['ds'])
print(f"Train: {len(train)} months | Test: {len(test)} months")

## 2. Fit Prophet Model

In [ ]:
model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                seasonality_mode='multiplicative')
model.fit(train)
print("Prophet model fitted")

## 3. Predict on Test Set

In [ ]:
future = model.make_future_dataframe(periods=12, freq='MS')
forecast = model.predict(future)

test_pred = forecast[forecast['ds'] >= '2014-01-01'][['ds','yhat','yhat_lower','yhat_upper']].reset_index(drop=True)
y_true = test['y'].values
y_pred = test_pred['yhat'].values

mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"Prophet Test Metrics:")
print(f"  MAPE:  {mape:.2f}%")
print(f"  MAE:   ${mae:,.2f}")
print(f"  RMSE:  ${rmse:,.2f}")
print(f"  R2:    {r2:.4f}")

## 4. Prophet Components

In [ ]:
fig = model.plot_components(forecast)
plt.savefig('../visualizations/prophet_components.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Visualize Predictions

In [ ]:
plt.figure(figsize=(14,6))
plt.plot(train['ds'], train['y'], label='Train', color=COLORS['primary'], linewidth=2)
plt.plot(test['ds'], y_true, label='Actual (Test)', color='black', linewidth=2, marker='o')
plt.plot(test['ds'], y_pred, label=f'Prophet Pred (MAPE={mape:.1f}%)', color=COLORS['success'], linestyle='--', linewidth=2, marker='d')
plt.fill_between(test['ds'], test_pred['yhat_lower'].values, test_pred['yhat_upper'].values, alpha=0.15, color=COLORS['success'])
plt.title('Prophet Forecast vs Actuals', fontsize=15, fontweight='bold')
plt.xlabel('Date'); plt.ylabel('Monthly Sales ($)'); plt.legend()
plt.tight_layout()
plt.savefig('../visualizations/prophet_forecast.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Save Model & Predictions

In [ ]:
pred_df = pd.DataFrame({'ds': test['ds'].values, 'actual': y_true, 'predicted': y_pred,
                        'lower': test_pred['yhat_lower'].values, 'upper': test_pred['yhat_upper'].values})
pred_df.to_csv('../predictions/prophet_predictions.csv', index=False)
joblib.dump(model, '../models/prophet_model.pkl')
print("Saved: predictions/prophet_predictions.csv")
print("Saved: models/prophet_model.pkl")
print("\nProceed to 04_XGBoost_Model.ipynb")